In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🗄️ Database Sharding & Replication Architecture Guide

**Scaling databases horizontally with sharding and ensuring high availability with replication**

This guide covers:
- Database sharding strategies
- Replication patterns (master-slave, multi-master)
- Consistency models (eventual, strong, causal)
- Backup and recovery
- Failover and disaster recovery
- Monitoring replicated systems
- Cost optimization
- Migration strategies
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Database Sharding Strategies

### Range-based Sharding
```python
class RangeShardRouter:
    """Route queries based on range of keys"""
    
    def __init__(self, shard_count: int, shard_size: int):
        self.shard_count = shard_count
        self.shard_size = shard_size
    
    def get_shard_id(self, user_id: int) -> int:
        """Calculate shard ID from user_id"""
        return (user_id - 1) // self.shard_size
    
    def get_shard_host(self, shard_id: int) -> str:
        """Get database host for shard"""
        shard_hosts = {
            0: "shard-db-1.aws.rds.us-east-1.amazonaws.com",
            1: "shard-db-2.aws.rds.us-east-1.amazonaws.com",
            2: "shard-db-3.aws.rds.us-east-1.amazonaws.com",
        }
        return shard_hosts.get(shard_id)

# Usage
router = RangeShardRouter(shard_count=3, shard_size=1_000_000)

user_id = 1_500_000
shard_id = router.get_shard_id(user_id)  # Returns 1
host = router.get_shard_host(shard_id)   # Returns shard-db-2
```

### Hash-based Sharding (Consistent Hashing)
```python
import hashlib
from typing import Dict, List

class ConsistentHashSharding:
    """Hash-based sharding with consistent hashing"""
    
    def __init__(self, nodes: List[str], virtual_nodes: int = 150):
        self.nodes = nodes
        self.virtual_nodes = virtual_nodes
        self.ring = {}
        self.sorted_keys = []
        self._build_ring()
    
    def _hash(self, key: str) -> int:
        """Hash function"""
        return int(hashlib.md5(key.encode()).hexdigest(), 16)
    
    def _build_ring(self):
        """Build consistent hash ring"""
        self.ring = {}
        
        for node in self.nodes:
            for i in range(self.virtual_nodes):
                virtual_key = f"{node}:{i}"
                hash_key = self._hash(virtual_key)
                self.ring[hash_key] = node
        
        self.sorted_keys = sorted(self.ring.keys())
    
    def get_node(self, key: str) -> str:
        """Get node for key"""
        if not self.ring:
            return None
        
        hash_key = self._hash(key)
        
        # Find node with smallest hash >= hash_key
        for ring_key in self.sorted_keys:
            if ring_key >= hash_key:
                return self.ring[ring_key]
        
        # Wrap around to first node
        return self.ring[self.sorted_keys[0]]
    
    def add_node(self, node: str):
        """Add new node and rebuild ring"""
        self.nodes.append(node)
        self._build_ring()
    
    def remove_node(self, node: str):
        """Remove node and rebuild ring"""
        self.nodes.remove(node)
        self._build_ring()

# Usage
sharding = ConsistentHashSharding(['db-1', 'db-2', 'db-3'])

shard = sharding.get_node('user:12345')  # Returns 'db-2'
```

### Directory-based Sharding
```sql
-- Sharding lookup table
CREATE TABLE shard_mapping (
    entity_id BIGINT PRIMARY KEY,
    shard_id INT NOT NULL,
    entity_type VARCHAR(50),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Index for fast lookups
CREATE INDEX idx_shard_mapping_entity ON shard_mapping(entity_id, shard_id);

-- Query to find shard
SELECT shard_id FROM shard_mapping WHERE entity_id = 123456;
```

### Python Implementation
```python
class ShardedDatabase:
    """Database access layer with automatic sharding"""
    
    def __init__(self, shard_hosts: Dict[int, str]):
        self.shard_hosts = shard_hosts
        self.connections = {}
    
    def _get_connection(self, shard_id: int):
        """Get or create connection to shard"""
        if shard_id not in self.connections:
            host = self.shard_hosts[shard_id]
            # Connection would be created here
            self.connections[shard_id] = f"connection_to_{host}"
        
        return self.connections[shard_id]
    
    def _calculate_shard(self, user_id: int) -> int:
        """Calculate shard ID"""
        return user_id % len(self.shard_hosts)
    
    def insert_user(self, user_id: int, name: str, email: str):
        """Insert user into appropriate shard"""
        shard_id = self._calculate_shard(user_id)
        conn = self._get_connection(shard_id)
        
        # Execute: INSERT INTO users (id, name, email) VALUES (?, ?, ?)
        return f"Inserted to shard {shard_id}"
    
    def get_user(self, user_id: int):
        """Get user from appropriate shard"""
        shard_id = self._calculate_shard(user_id)
        conn = self._get_connection(shard_id)
        
        # Execute: SELECT * FROM users WHERE id = ?
        return f"Query shard {shard_id}"
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Replication Patterns

### Master-Slave (Primary-Replica) Replication
```
Write Flow:
Client → Master (writes) → Async replication → Slave (read-only)

Advantages:
- Simple setup
- Good for read-heavy workloads
- Easy failover

Disadvantages:
- Master is single point of write
- Replication lag can cause stale reads
```

### Master-Master (Multi-Master) Replication
```
Write Flow:
Client → Master 1 (writes)  ← → Master 2 (writes)
         ↓ replication          ↓ replication
       Slave 1                Slave 2

Advantages:
- Active-active setup
- No single write point of failure
- Lower latency for writes

Disadvantages:
- Complex conflict resolution
- Higher operational overhead
- Risk of split-brain scenarios
```

### PostgreSQL Replication Setup
```bash
#!/bin/bash
# setup-pg-replication.sh

# On Master Server (Primary)
# 1. Enable replication in postgresql.conf
sed -i "s/^#wal_level = replica/wal_level = replica/" /etc/postgresql/14/main/postgresql.conf
sed -i "s/^#max_wal_senders = 10/max_wal_senders = 10/" /etc/postgresql/14/main/postgresql.conf
sed -i "s/^#max_replication_slots = 10/max_replication_slots = 10/" /etc/postgresql/14/main/postgresql.conf

# 2. Restart PostgreSQL
sudo systemctl restart postgresql

# 3. Create replication user
sudo -u postgres psql << EOF
CREATE ROLE replicator WITH REPLICATION LOGIN PASSWORD 'password';
EOF

# 4. Set up pg_hba.conf for replication
echo "host replication replicator 192.168.1.0/24 md5" | sudo tee -a /etc/postgresql/14/main/pg_hba.conf

# On Slave Server (Replica)
# 1. Stop PostgreSQL
sudo systemctl stop postgresql

# 2. Backup master data
rm -rf /var/lib/postgresql/14/main
pg_basebackup -h master_ip -D /var/lib/postgresql/14/main -U replicator -v -P

# 3. Enable standby mode
echo "standby_mode = 'on'" | sudo tee /var/lib/postgresql/14/main/recovery.conf

# 4. Start PostgreSQL
sudo systemctl start postgresql

# Verify replication
sudo -u postgres psql -c "SELECT * FROM pg_stat_replication;"
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Consistency Models

### Eventual Consistency
```python
# Example: Caching with eventual consistency

import asyncio
import redis

async def get_user_eventual_consistency(user_id: int, db_conn, cache: redis.Redis):
    """
    Try cache first, then fall back to database
    Cache is updated asynchronously
    """
    
    # Check cache
    cached = cache.get(f"user:{user_id}")
    if cached:
        return json.loads(cached)
    
    # Not in cache, query database
    user = await db_conn.get_user(user_id)
    
    # Update cache asynchronously (fire-and-forget)
    asyncio.create_task(cache.setex(
        f"user:{user_id}",
        3600,  # 1 hour TTL
        json.dumps(user)
    ))
    
    return user
```

### Strong Consistency
```python
# Two-phase commit for strong consistency

class TwoPhaseCommitManager:
    def __init__(self, coordinator, participants):
        self.coordinator = coordinator
        self.participants = participants
    
    async def execute_transaction(self, transaction):
        """Execute with two-phase commit"""
        
        # Phase 1: Prepare
        votes = []
        for participant in self.participants:
            can_commit = await participant.prepare(transaction)
            votes.append(can_commit)
        
        if all(votes):
            # Phase 2: Commit
            for participant in self.participants:
                await participant.commit(transaction)
            return True
        else:
            # Rollback
            for participant in self.participants:
                await participant.rollback(transaction)
            return False
```

### Causal Consistency
```python
# Vector clocks for causal consistency

class VectorClock:
    def __init__(self, node_id: str):
        self.node_id = node_id
        self.clock = {}
    
    def increment(self):
        """Increment local clock"""
        self.clock[self.node_id] = self.clock.get(self.node_id, 0) + 1
    
    def update(self, other_clock: dict):
        """Update with received clock"""
        for node, time in other_clock.items():
            self.clock[node] = max(self.clock.get(node, 0), time)
        self.increment()
    
    def happens_before(self, other: 'VectorClock') -> bool:
        """Check causal order"""
        less_or_equal = all(
            self.clock.get(k, 0) <= other.clock.get(k, 0)
            for k in set(list(self.clock.keys()) + list(other.clock.keys()))
        )
        strictly_less = any(
            self.clock.get(k, 0) < other.clock.get(k, 0)
            for k in other.clock
        )
        return less_or_equal and strictly_less
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 4. Backup & Recovery

### Automated Backup Strategy
```python
import backup_manager

class DatabaseBackupManager:
    def __init__(self, db_host: str, backup_interval_hours: int = 24):
        self.db_host = db_host
        self.backup_interval = backup_interval_hours
    
    def create_backup(self, backup_type: str = "full"):
        """Create database backup"""
        if backup_type == "full":
            # Full backup
            return backup_manager.pg_dump(self.db_host)
        elif backup_type == "incremental":
            # Incremental backup (WAL)
            return backup_manager.wal_backup(self.db_host)
    
    def restore_from_backup(self, backup_path: str, point_in_time: datetime = None):
        """Restore database"""
        if point_in_time:
            # Point-in-time recovery
            return backup_manager.pitr_restore(backup_path, point_in_time)
        else:
            # Full restore
            return backup_manager.pg_restore(backup_path)
    
    def verify_backup(self, backup_path: str) -> bool:
        """Verify backup integrity"""
        return backup_manager.verify_integrity(backup_path)

# Usage
backup_mgr = DatabaseBackupManager("db.example.com")
backup_mgr.create_backup("full")  # Weekly full backup
backup_mgr.create_backup("incremental")  # Daily incremental
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 5. Database Scaling Checklist

✅ **Sharding**
- [ ] Sharding key selected (avoid hotspots)
- [ ] Consistent hashing implemented
- [ ] Cross-shard queries minimized
- [ ] Distributed joins handled
- [ ] Rebalancing strategy defined

✅ **Replication**
- [ ] Read replicas provisioned
- [ ] Replication lag monitored
- [ ] Failover procedure tested
- [ ] Backups automated
- [ ] Recovery tested monthly

✅ **Consistency**
- [ ] Consistency model chosen
- [ ] Application handles eventual consistency
- [ ] Conflict resolution strategy
- [ ] Read-after-write consistency ensured
- [ ] Monitoring for anomalies

✅ **Operations**
- [ ] Monitoring dashboards set up
- [ ] Alerts configured for replication lag
- [ ] Backup retention policy defined
- [ ] Disaster recovery plan documented
- [ ] Regular DR drills scheduled
</MySCode.Cell>
```